In [0]:
%run ../../utils/00_silver_utils

In [0]:
# Widget params; run_id comes from bronze via get_shared_run_id so this notebook's audit rows share bronze's run_id.
dbutils.widgets.text("silver_config_path", "/Volumes/cdr_silver/config/silver_config/silver_config.json")
dbutils.widgets.text("table_filter", "")

silver_config_path = dbutils.widgets.get("silver_config_path")
table_filter_raw = dbutils.widgets.get("table_filter").strip()
table_filter = {t.strip().lower() for t in table_filter_raw.split(",") if t.strip()} if table_filter_raw else None

dbutils.widgets.text("run_id", "")
run_id_param = dbutils.widgets.get("run_id").strip()
run_id = run_id_param if run_id_param else str(uuid.uuid4())
if not run_id_param:
    logger.warning(f"No run_id widget value supplied -- generating standalone run_id={run_id}")

logger.info(f"run_id              = {run_id}")
logger.info(f"silver_config_path  = {silver_config_path}")
logger.info(f"table_filter        = {table_filter or 'ALL TABLES FOR THIS NOTEBOOK'}")

In [0]:
# Loads silver_config.json and filters it down to the tables tagged silver_notebook == "orders_meds_icu".
with open(silver_config_path, "r") as f:
    SILVER_CONFIG = json.load(f)

LAYER = "orders_meds_icu"
tables_for_this_notebook = [
    t for t, cfg in SILVER_CONFIG.items() if cfg["silver_notebook"] == LAYER
]
if table_filter:
    tables_for_this_notebook = [t for t in tables_for_this_notebook if t in table_filter]

logger.info(f"Processing {len(tables_for_this_notebook)} table(s) for layer '{LAYER}': {tables_for_this_notebook}")

ensure_silver_audit_table(spark)

In [0]:
# Runs every table for this layer: read bronze, clean, write/merge to silver, collect an audit row, then write the audit log.
audit_rows = []
success_count = 0
failure_count = 0

for table_name in tables_for_this_notebook:
    audit_row = process_table(
        spark, table_name, SILVER_CONFIG, run_id, layer=LAYER,
        special_transform=None,
    )
    audit_rows.append(audit_row)
    if audit_row.status == "SUCCESS":
        success_count += 1
    else:
        failure_count += 1

write_audit_rows(spark, audit_rows)

logger.info(
    f"Layer '{LAYER}' complete | run_id={run_id} | {success_count} succeeded | "
    f"{failure_count} failed | {len(tables_for_this_notebook)} total tables"
)
if failure_count > 0:
    logger.warning(f"{failure_count} table(s) failed -- check {SILVER_AUDIT_TABLE} (run_id = {run_id})")